# Beyond Thought Anchors — Reproduce Whitebox Methods

**Method 2**: Receiver Head Attention Analysis  
**Method 3**: Causal Masking (Sentence-level KL Suppression)  

Model: `qwen-14b` = `deepseek-ai/DeepSeek-R1-Distill-Qwen-14B` (48 layers, 40 heads)  
Datasets: GPQA (20 questions) + Math (20 questions), correct base CoT only

**Output files:**
- Method 2 GPQA: `csvs/gpqa/receiver_head_scores_correct_qwen-14b_k32_pi16.csv`
- Method 2 Math: `csvs/math/receiver_head_scores_correct_qwen-14b_k32_pi16.csv`
- Method 3 GPQA: `kl_results/gpqa/qwen-14b/correct/problem_N_kl.npy`
- Method 3 Math: `kl_results/math/qwen-14b/correct/problem_N_kl.npy`

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.__version__)

!nvidia-smi

In [ ]:
import os

os.chdir("/content/drive/MyDrive/Beyond-Thought-Anchors/")
print(os.getcwd())

!pip install -r requirements.txt

---
## Method 2: Receiver Head Attention Analysis

**Step 1** `prep_attn_cache.py` — 加载 14B 模型，对每道题做 **1次** forward pass，一次性提取全部 48层×40头 的 attention 矩阵，存入 `attn_cache/`（1920个 `.npy` 文件/题）。

**Step 2** `generate_rec_csvs.py` — 直接读 `.npy` 缓存，计算 receiver head scores，输出 CSV。不再过模型。

> Step 1 完成后断线重连，Step 2 可直接运行（缓存已在 Drive 上）。

### Method 2 — GPQA

In [ ]:
# Step 1: Cache attention matrices for GPQA (1次 forward pass/题)
!python whitebox-analyses/scripts/prep_attn_cache.py \
    --model qwen-14b \
    --dataset gpqa \
    --skip-receiver

In [ ]:
# Step 2: Generate receiver head CSV for GPQA
# 预计时长：~10s（缓存已就绪）
# 输出：csvs/gpqa/receiver_head_scores_correct_qwen-14b_k32_pi16.csv
!python whitebox-analyses/scripts/generate_rec_csvs.py \
    --model-name qwen-14b \
    --data-model qwen-14b \
    --dataset gpqa \
    --top-k 32 \
    --proximity-ignore 16 \
    --output-dir csvs/gpqa

### Method 2 — Math

In [ ]:
# Step 1: Cache attention matrices for Math
!python whitebox-analyses/scripts/prep_attn_cache.py \
    --model qwen-14b \
    --dataset math \
    --skip-receiver

In [ ]:
# Step 2: Generate receiver head CSV for Math
# 输出：csvs/math/receiver_head_scores_correct_qwen-14b_k32_pi16.csv
!python whitebox-analyses/scripts/generate_rec_csvs.py \
    --model-name qwen-14b \
    --data-model qwen-14b \
    --dataset math \
    --top-k 32 \
    --proximity-ignore 16 \
    --output-dir csvs/math

---
## Method 3: Causal Masking (KL Suppression)

对每道题的每个句子，mask 掉该句的 attention，重新做 forward pass，计算 token 级 KL 散度，得到句×句因果影响矩阵。

**每道题需要 N_句次 forward pass**（GPQA 平均约 100句/题）。

**输出**: `kl_results/{dataset}/qwen-14b/correct/problem_N_kl.npy`（句×句 numpy 矩阵）

> 行求和 → per-chunk causal importance（Exp 3 方法三排名）。
>
> 断线恢复：`@pkld` 会自动跳过已完成的题，重跑不会重算。

### Method 3 — GPQA

In [ ]:
# 输出：kl_results/gpqa/qwen-14b/correct/problem_N_kl.npy
!python whitebox-analyses/scripts/prep_suppression_mtxs.py \
    --model-name qwen-14b \
    --dataset gpqa \
    --output-dir kl_results/gpqa

### Method 3 — Math

In [ ]:
# 输出：kl_results/math/qwen-14b/correct/problem_N_kl.npy
!python whitebox-analyses/scripts/prep_suppression_mtxs.py \
    --model-name qwen-14b \
    --dataset math \
    --output-dir kl_results/math

---
## 验证输出文件

In [ ]:
import os
import numpy as np
import pandas as pd

print("=== Method 2: Receiver Head CSVs ===")
for dataset in ["gpqa", "math"]:
    path = f"csvs/{dataset}/receiver_head_scores_correct_qwen-14b_k32_pi16.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)
        n_problems = df["problem_number"].nunique()
        print(f"  [{dataset}] {path} — {n_problems} problems, {len(df)} sentences")
    else:
        print(f"  [{dataset}] MISSING: {path}")

print()
print("=== Method 3: KL Suppression Matrices ===")
for dataset in ["gpqa", "math"]:
    kl_dir = f"kl_results/{dataset}/qwen-14b/correct"
    if os.path.exists(kl_dir):
        files = [f for f in os.listdir(kl_dir) if f.endswith(".npy")]
        print(f"  [{dataset}] {kl_dir} — {len(files)} matrices")
        for f in sorted(files)[:3]:
            mat = np.load(os.path.join(kl_dir, f))
            print(f"    {f}: shape={mat.shape}")
    else:
        print(f"  [{dataset}] MISSING: {kl_dir}")